Перед запуском убедитесь, что в корне проекта есть файл .env и в нем заполнены выданные вам креды подключения к базам данных и хранилищу

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine

In [3]:
# подгружаем .env
load_dotenv()

True

In [4]:
# Считываем все креды
src_host = os.environ.get('DB_SOURCE_HOST')
src_port = os.environ.get('DB_SOURCE_PORT')
src_username = os.environ.get('DB_SOURCE_USER')
src_password = os.environ.get('DB_SOURCE_PASSWORD')
src_db = os.environ.get('DB_SOURCE_NAME') 

dst_host = os.environ.get('DB_DESTINATION_HOST')
dst_port = os.environ.get('DB_DESTINATION_PORT')
dst_username = os.environ.get('DB_DESTINATION_USER')
dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
dst_db = os.environ.get('DB_DESTINATION_NAME')

s3_bucket = os.environ.get('S3_BUCKET_NAME')
s3_access_key = os.environ.get('AWS_ACCESS_KEY_ID')
s3_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')

In [5]:
# Создадим соединения
src_conn = create_engine(f'postgresql://{src_username}:{src_password}@{src_host}:{src_port}/{src_db}')
dst_conn = create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')

In [ ]:
TABLE = 'data_combined'
SQL = f'select * from {TABLE}'
data = pd.read_sql(SQL, dst_conn)

Нужно убедиться что в данных нет пропусков, исходя из этой таблицы там их нет

In [7]:
cols_with_nans = data.isnull().sum()
print(cols_with_nans)

flat_id              0
building_id          0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
target               0
dtype: int64


Проверка на то что таблице нет дубликатов. Сверяюсь по всем столбцам 

In [10]:
print(data.dtypes)


flat_id                int64
building_id            int64
floor                  int64
kitchen_area         float64
living_area          float64
rooms                  int64
is_apartment            bool
studio                  bool
total_area           float64
price                  int64
build_year             int64
building_type_int      int64
latitude             float64
longitude            float64
ceiling_height       float64
flats_count            int64
floors_total           int64
has_elevator            bool
target                 int64
dtype: object


In [8]:
is_duplicated_id = data.duplicated(subset=data.columns.to_list(), keep=False)
print(sum(is_duplicated_id)) 

0


In [11]:
cols_to_check = [
        "target",
        "kitchen_area",
        "living_area",
        "total_area",
        "price",
        "ceiling_height",
        "rooms",
        "floor",
        "flats_count",
        "floors_total",
        "build_year",
    ]

# num_cols = data.select_dtypes(['float']).columns
threshold = 1.5
potential_outliers = pd.DataFrame()

for col in cols_to_check:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    margin = threshold*IQR
    lower = Q1 - margin
    upper = Q3 + margin
    potential_outliers[col] = ~data[col].between(lower, upper)

outliers = potential_outliers.any(axis=1)


In [12]:
outliers_data = data[outliers]
print(f"Количество строк с аномалиями: {len(outliers_data)}")
print(outliers_data)

cleaned_data = data[~outliers] 
print(f"Количество чистых строк для обучения: {len(cleaned_data)}")

Количество строк с аномалиями: 29182
        flat_id  building_id  floor  kitchen_area  living_area  rooms  \
3             3        18579      1          10.1    43.099998      3   
41           41        21812     24          14.0    20.000000      1   
44           44        23605      9           0.0     0.000000      1   
48           48        19189      8          14.0    64.800003      3   
49           49        22115     22          12.0    60.000000      2   
...         ...          ...    ...           ...          ...    ...   
141341   141341        20778     27          16.0    73.000000      3   
141344   141344        19578     14          17.1    80.199997      3   
141345   141345        22477     12          12.0    53.799999      3   
141352   141352        10589      1           7.5    19.299999      1   
141357   141357        22455     16          11.0    18.000000      1   

        is_apartment  studio  total_area     price  build_year  \
3              False

А дальше, творите!